In [ ]:
import myFunctions as myF
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

# Set font configurations for Matplotlib (Times New Roman for English, SimHei for Chinese)
plt.rcParams['font.family'] = ['Times New Roman', 'SimHei']
plt.rcParams['axes.unicode_minus'] = False

# Note: Loading only the first 88,000 rows as per original script
df = pd.read_csv('./merged_combined_data.csv').iloc[:88000]

# Feature Engineering: Extract Month and Day
df['转化月份和日'] = pd.to_datetime(df['日期'], format='ISO8601')
df['月份'] = df['转化月份和日'].dt.month
df['日'] = df['转化月份和日'].dt.day
df.drop(columns=['转化月份和日'], inplace=True)

# Visualization: Price Distribution 
plt.figure(figsize=(10, 6), dpi=400)
sns.histplot(df['实时出清价格(元/MWh)'], kde=True, bins='auto', stat='probability',
             color=plt.cm.Blues(0.5))
# plt.title('Distribution of "Real-time Clearing Price (CNY/MWh)" in Matching Data Set', fontsize=14)
plt.xlabel('Real-time clearing price (CNY/MWh)', fontsize=24)
plt.ylabel('Proportion', fontsize=24)
plt.xlim(-20, 1520)
plt.xticks([0, 200, 400, 600, 900, 1200, 1500])
# Adjust tick label size
plt.tick_params(axis='both', which='major', labelsize=24)
plt.tight_layout()
plt.show()

# --- QuantileLSTM Forecasting Experiments ---

In [ ]:
# Define base columns for LSTM models
columns = ['序号', '非市场化机组出力(MW)', '水电实际值(MW)', '风电实际值(MW)', '光伏实际值(MW)',
           '新能源总加实际值(MW)', '实时出清价格(元/MWh)', '日前出清价格(元/MWh)', '检修容量(MW)',
           '日前联络线电力值(MW)', '省内负荷预测电力值(MW)', '日前联络线和省内负荷预测总加电力值(MW)',
           '统调用电负荷电力值(MW)', '新能源出力预测电力值(MW)', '新能源出力预测风电(MW)', '新能源出力预测光伏(MW)',
           't2m', 'u10', 'v10', 'tp', '月份', '日']

## Experiment 1: Basic LSTM-QR Usage

In [ ]:
data = df[columns].copy()
print(f"Basic LSTM Data Shape: {data.shape}")

# Preprocess time column
df['时点'] = df['时点'].replace('24:00', '23:59')
data['_date'] = pd.to_datetime(df['日期'] + ' ' + df['时点'], format='ISO8601')

# Run QuantileLSTM
result_json = myF.QuantileLSTMAnalysis().run_time(
    data_with_time=data.copy().dropna(), 
    time_name='_date', 
    name='实时出清价格(元/MWh)',
    seq_len=1*96, 
    batch_size=2048, 
    epochs=50, 
    test_column='_date', 
    test_range=['2024-04-01', '2024-06-01'],
    quantiles=[0.08, 0.92]
)

# Save results
output_file = f'./results/QuantileLSTM_results-20240401-84.json'
with open(output_file, 'w', encoding='utf-8') as f:
    f.write(result_json)
print(f"Results written to {output_file}")

## Experiment 2: LSTM-QR with Category (Clustering)

In [ ]:
data = df[columns].copy().dropna()

# Perform clustering on input data to create a 'category' feature
kmeans = KMeans(n_clusters=2, random_state=10, n_init=10) 
kmeans.fit(data)
data['category'] = kmeans.labels_

# Preprocess time
df['时点'] = df['时点'].replace('24:00', '23:59')
data['_date'] = pd.to_datetime(df['日期'] + ' ' + df['时点'], format='ISO8601')

# Run LSTM with categorical column
result_json = myF.QuantileLSTMAnalysis().run_time(
    data_with_time=data.copy().dropna(), 
    time_name='_date', 
    name='实时出清价格(元/MWh)',
    seq_len=1*96, 
    categorical_columns=['category'], 
    batch_size=2048, 
    epochs=50, 
    test_column='_date', 
    test_range=['2024-04-01', '2024-06-01'],
    quantiles=[0.08, 0.92]
)

# Save results
output_file = f'./results/QuantileLSTM_results-20240401-category-84.json'
with open(output_file, 'w', encoding='utf-8') as f:
    f.write(result_json)
print(f"Results written to {output_file}")

## Experiment 3: LSTM-QR with Feature Construction (FC)

In [ ]:
data = df[columns].copy()
print(f"FC Data Shape (Before): {data.shape}")

# Create lag features (shifting columns)
shift_columns = ['风电实际值(MW)', '光伏实际值(MW)', '日前出清价格(元/MWh)',
                 '省内负荷预测电力值(MW)', '统调用电负荷电力值(MW)',
                 '实时出清价格(元/MWh)',  '新能源出力预测电力值(MW)']
for col in shift_columns:
    for i in range(1, 5):
        data[f'{col}_{i}'] = data[f'{col}'].shift(i) 

print(f"FC Data Shape (After): {data.shape}")

# Preprocess time
df['时点'] = df['时点'].replace('24:00', '23:59')
data['_date'] = pd.to_datetime(df['日期'] + ' ' + df['时点'], format='ISO8601')

# Run LSTM
result_json = myF.QuantileLSTMAnalysis().run_time(
    data_with_time=data.copy().dropna(), 
    time_name='_date', 
    name='实时出清价格(元/MWh)',
    seq_len=1*96, 
    batch_size=1024, 
    epochs=50, 
    test_column='_date', 
    test_range=['2024-04-01', '2024-06-01'],
    quantiles=[0.08, 0.92]
)

# Save results
output_file = f'./results/QuantileLSTM_results-20240401-FC-84.json'
with open(output_file, 'w', encoding='utf-8') as f:
    f.write(result_json)
print(f"Results written to {output_file}")

## Experiment 4: LSTM-QR with Feature Selection (FS)

In [ ]:
data = df[columns].copy()

# Perform feature selection using Symmetric Uncertainty
selected_features = myF.SymmetricUncertainty().select_features(
    data=data.dropna(), 
    target_col='实时出清价格(元/MWh)'
)

print(f'Selected features: {selected_features}')

# Ensure target column is included
selected_features.append('实时出清价格(元/MWh)')
data = data[selected_features]
print(f"FS Data Shape: {data.shape}")

# Preprocess time
df['时点'] = df['时点'].replace('24:00', '23:59')
data['_date'] = pd.to_datetime(df['日期'] + ' ' + df['时点'], format='ISO8601')

# Run LSTM
result_json = myF.QuantileLSTMAnalysis().run_time(
    data_with_time=data.copy().dropna(), 
    time_name='_date', 
    name='实时出清价格(元/MWh)',
    seq_len=1*96, 
    batch_size=2048, 
    epochs=50, 
    test_column='_date', 
    test_range=['2024-04-01', '2024-06-01'],
    quantiles=[0.08, 0.92]
)

# Save results
output_file = f'./results/QuantileLSTM_results-20240401-FS-84.json'
with open(output_file, 'w', encoding='utf-8') as f:
    f.write(result_json)
print(f"Results written to {output_file}")

## Experiment 5: LSTM-QR with FC and FS

In [ ]:
data = df[columns].copy()

# 1. Feature Construction (FC)
shift_columns = ['风电实际值(MW)', '光伏实际值(MW)', '日前出清价格(元/MWh)',
                 '省内负荷预测电力值(MW)', '统调用电负荷电力值(MW)',
                 '实时出清价格(元/MWh)',  '新能源出力预测电力值(MW)']
for col in shift_columns:
    for i in range(1, 5):
        data[f'{col}_{i}'] = data[f'{col}'].shift(i) 

# 2. Feature Selection (FS)
selected_features = myF.SymmetricUncertainty().select_features(
    data=data.dropna(), 
    target_col='实时出清价格(元/MWh)'
)
print(f'Selected features: {selected_features}')

selected_features.append('实时出清价格(元/MWh)')
data = data[selected_features]
print(f"FC-FS Data Shape: {data.shape}")

# Preprocess time
df['时点'] = df['时点'].replace('24:00', '23:59')
data['_date'] = pd.to_datetime(df['日期'] + ' ' + df['时点'], format='ISO8601')

# Run LSTM
result_json = myF.QuantileLSTMAnalysis().run_time(
    data_with_time=data.copy().dropna(), 
    time_name='_date', 
    name='实时出清价格(元/MWh)',
    seq_len=1*96, 
    batch_size=2048, 
    epochs=50, 
    test_column='_date', 
    test_range=['2024-04-01', '2024-06-01'],
    quantiles=[0.08, 0.92]
)

# Save results
output_file = f'./results/QuantileLSTM_results-20240401-FC-FS-84.json'
with open(output_file, 'w', encoding='utf-8') as f:
    f.write(result_json)
print(f"Results written to {output_file}")